## Mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Import packages/install libraries

In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import torchvision.transforms as T

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

## Data Preprocessing

In [ ]:
# Using GPU cuz CPU takes too damn long
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Define transformation

# Add augmentation for training
train_transformation = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Affine(
        translate_percent=0.05,
        scale=(0.9, 1.1),
        rotate=0,
        p=0.5,
        interpolation=cv2.INTER_LINEAR,      # for image
        mask_interpolation=cv2.INTER_NEAREST
    ),
    A.HueSaturationValue(p=0.3),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
    A.ElasticTransform(p=0.2),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

transformation = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

/tmp/ipykernel_4705/2588745875.py:16: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),


### Define Dataset Class

In [ ]:
class GingivitisDataset(Dataset):
    def __init__(self, image_folder, mask_folder, transform=None):
        self.image_folder = Path(image_folder)
        self.mask_folder = Path(mask_folder)
        self.transform = transform

        # Get all images
        self.image_paths = sorted(self.image_folder.glob("*.jpg"))

        # Match masks directly
        self.mask_paths = [
            self.mask_folder / f"{img_path.stem}.png"
            for img_path in self.image_paths
        ]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]
        image_name = img_path.name

        # Load image (RGB)
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Load mask (already class indices!)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        # Safety check
        if mask is None:
            raise FileNotFoundError(
                f"Mask not found for {image_name} at {mask_path}"
            )

        mask = mask.astype(np.uint8)

        # Optional sanity check (can remove later)
        if not np.any(mask != 255):
            print(f"Warning: Empty mask (all ignore) in {image_name}")

        # Apply transforms
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"].long()

        return image, mask, image_name

### Create dataset objects

In [ ]:
train_dataset = GingivitisDataset(
    image_folder="/content/drive/MyDrive/CSS2/CSS2_480p (Halved)/Dataset/Training/Images",
    mask_folder="/content/drive/MyDrive/CSS2/CSS2_480p (Halved)/Dataset/Training/SAM_MasksIndex",
    transform=train_transformation
)

val_dataset = GingivitisDataset(
    image_folder="/content/drive/MyDrive/CSS2/CSS2_480p (Halved)/Dataset/Validation/Images",
    mask_folder="/content/drive/MyDrive/CSS2/CSS2_480p (Halved)/Dataset/Validation/SAM_MasksIndex",
    transform=transformation
)

test_dataset = GingivitisDataset(
    image_folder="/content/drive/MyDrive/CSS2/CSS2_480p (Halved)/Dataset/Test/Images",
    mask_folder="/content/drive/MyDrive/CSS2/CSS2_480p (Halved)/Dataset/Test/SAM_MasksIndex",
    transform=transformation
)

## Create DataLoaders

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory= True) # shuffle is true for training so model learns instead of memorising; pin_memory is to transfer from cpu to gpu faster
val_loader   = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory= True)
test_loader  = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory= True)

## Load PAN (Pyramid Attention Network)

In [ ]:
model = smp.PAN(
    encoder_name="efficientnet-b0",
    encoder_weights="imagenet",
    in_channels=3, # RGB channels
    classes=5, # 5 classes
    encoder_output_stride=16,
    decoder_channels=32,
)

model = model.to(device) # using GPU

# Freeze encoder for first 5 epochs
for param in model.encoder.parameters():
    param.requires_grad = False

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [ ]:
# Loss function and optimizer
weights = torch.tensor([15.0, 4.0, 3.5, 2.5, 4.0], dtype=torch.float).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-3)

## Training

In [ ]:
num_epochs = 40
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
)
best_val_loss = float('inf')
patience = 12
patience_counter = 0
ce_loss   = torch.nn.CrossEntropyLoss(weight=weights, ignore_index=255)
dice_loss = smp.losses.DiceLoss(mode='multiclass', ignore_index=255)

for epoch in range(num_epochs):

  # Unfreeze encoder after epoch 5
  if epoch == 5:
      for param in model.encoder.parameters():
          param.requires_grad = True
      print('Encoder unfrozen')

  if epoch == 10:
    # Unfreeze everything
    for param in model.encoder.parameters():
        param.requires_grad = True
    print('Encoder fully unfrozen')

  # Training loop
  model.train()
  total_loss = 0
  train_batches = 0

  for i, (images, masks, names) in enumerate(train_loader):
    images = images.to(device)   # using GPU
    masks = masks.to(device)

    # Skip batch if there are no valid pixels in the mask for loss calculation
    if not (masks != 255).any():
      print(f"Training Skipping Batch {i}, images: {names} since all mask pixels are ignore_index (255)")
      continue

    outputs = model(images)
    loss = ce_loss(outputs, masks) + dice_loss(outputs, masks)

    optimizer.zero_grad() # clear old graidents from previous steps
    loss.backward() # backward pass to calculate the gradients for all parameters
    optimizer.step() # update model

    total_loss += loss.item()
    train_batches += 1

  # Validation loop
  model.eval() # set model to evaluation mode
  val_loss = 0
  val_batches = 0

  with torch.no_grad():
    for i, (images, masks, names) in enumerate(val_loader):
      images = images.to(device)
      masks = masks.to(device)

      # Skip batch if there are no valid pixels in the mask for loss calculation
      if not (masks != 255).any():
        print(f"Validation Skipping Batch {i}, images: {names} since all mask pixels are ignore_index (255).")
        continue # Skip this batch

      outputs = model(images)
      loss = ce_loss(outputs, masks) + dice_loss(outputs, masks)
      val_loss += loss.item()
      val_batches += 1

  train_loss = total_loss / train_batches if train_batches > 0 else 0
  val_loss = val_loss / val_batches if val_batches > 0 else 0

  scheduler.step(val_loss)

  print(f"Epoch {epoch+1}")
  print(f"Train loss: {train_loss:.2f}")
  print(f"Val loss: {val_loss:.2f}")

  if val_loss < best_val_loss:
      best_val_loss = val_loss
      patience_counter = 0
      torch.save(model.state_dict(), '/content/pan_sam_best.pth')
      print('Improvement')
  else:
      patience_counter += 1
      print(f'No improvement ({patience_counter}/{patience})')
      if patience_counter >= patience:
          print('Early stopping triggered')
          break

Epoch 1
Train loss: 1.70
Val loss: 2.03
Improvement
Epoch 2
Train loss: 1.46
Val loss: 2.13
No improvement (1/12)
Epoch 3
Train loss: 1.43
Val loss: 2.31
No improvement (2/12)
Epoch 4
Train loss: 1.38
Val loss: 2.16
No improvement (3/12)
Epoch 5
Train loss: 1.37
Val loss: 2.08
No improvement (4/12)
Encoder unfrozen
Epoch 6
Train loss: 1.33
Val loss: 2.07
No improvement (5/12)
Epoch 7
Train loss: 1.33
Val loss: 1.96
Improvement
Epoch 8
Train loss: 1.34
Val loss: 1.96
Improvement
Epoch 9
Train loss: 1.33
Val loss: 2.15
No improvement (1/12)
Epoch 10
Train loss: 1.32
Val loss: 1.96
No improvement (2/12)
Encoder fully unfrozen
Epoch 11
Train loss: 1.34
Val loss: 1.98
No improvement (3/12)
Epoch 12
Train loss: 1.34
Val loss: 2.22
No improvement (4/12)
Epoch 13
Train loss: 1.31
Val loss: 1.98
No improvement (5/12)
Epoch 14
Train loss: 1.35
Val loss: 1.94
Improvement
Epoch 15
Train loss: 1.31
Val loss: 2.04
No improvement (1/12)
Epoch 16
Train loss: 1.32
Val loss: 2.10
No improvement (2/12)
E

KeyboardInterrupt: 

## Testing

IoU checks how much the prediction overlaps with the ground truth (correct labels).  
Dice measures similarity between prediction and ground truth but gives more weight to overlapping pixels.  
Calculating the mean for these two to show one score overall for them.

In [ ]:
def evaluate(model, dataloader, device, num_classes=5):
  model.load_state_dict(torch.load('/content/pan_sam_best.pth'))
  model.eval()

  total_correct = 0
  total_pixels = 0

  # Accumulate per-class stats (dataset-level)
  intersection_per_class = [0] * num_classes
  union_per_class = [0] * num_classes
  dice_num = [0] * num_classes
  dice_den = [0] * num_classes
  tp_per_class = [0] * num_classes
  fn_per_class = [0] * num_classes  # <-- fixed indent

  with torch.no_grad():
    for images, masks, names in dataloader:
      images = images.to(device)
      masks = masks.to(device)

      outputs = model(images)
      preds = torch.argmax(outputs, dim=1)

      valid = masks != 255  # ignore background

      # Accuracy
      correct = (preds == masks) & valid
      total_correct += correct.sum().item()
      total_pixels += valid.sum().item()

      # Per-class metrics
      for cls in range(num_classes):
        pred_cls = (preds == cls) & valid
        mask_cls = (masks == cls) & valid

        intersection = (pred_cls & mask_cls).sum().item()
        union = (pred_cls | mask_cls).sum().item()

        # IoU
        intersection_per_class[cls] += intersection
        union_per_class[cls] += union

        # Dice
        dice_num[cls] += 2 * intersection
        dice_den[cls] += pred_cls.sum().item() + mask_cls.sum().item()

        # Recall (TP and FN)
        tp_per_class[cls] += intersection
        fn_per_class[cls] += (mask_cls & (~pred_cls)).sum().item()

  # Accuracy
  accuracy = total_correct / total_pixels if total_pixels > 0 else 0

  # Mean IoU
  iou_scores = []
  for c in range(num_classes):
    if union_per_class[c] > 0:
      iou_scores.append(intersection_per_class[c] / union_per_class[c])
  mean_iou = sum(iou_scores) / len(iou_scores) if iou_scores else 0

  # Mean Dice
  dice_scores = []
  for c in range(num_classes):
    if dice_den[c] > 0:
      dice_scores.append(dice_num[c] / (dice_den[c] + 1e-6))
  mean_dice = sum(dice_scores) / len(dice_scores) if dice_scores else 0

  # Mean Recall
  recall_scores = []
  for c in range(num_classes):
    tp = tp_per_class[c]
    fn = fn_per_class[c]
    if (tp + fn) > 0:
      recall_scores.append(tp / (tp + fn))
  mean_recall = sum(recall_scores) / len(recall_scores) if recall_scores else 0

  return accuracy, mean_iou, mean_dice, mean_recall

In [ ]:
test_acc, test_iou, test_dice, test_recall = evaluate(model, test_loader, device)

print(f"Test Accuracy: {test_acc:.2f}")
print(f"Mean IoU: {test_iou:.2f}")
print(f"Mean Dice: {test_dice:.2f}")
print(f"Mean Recall: {test_recall:.2f}")

**Results**  
Note: Using ResNet34 would make PAN middle weight  
possible ResNet18

**PAN + ResNet18**  (had balanced cropping)  
Test Accuracy: 0.37  
Mean IoU: 0.10  
Mean Dice: 0.15  

**PAN + EfficientnetB0**  
Test Accuracy: 0.46  
Mean IoU: 0.24  
Mean Dice: 0.34

**PAN + MobileNetV2**  
Test Accuracy: 0.39  
Mean IoU and Dice were < 0.2

**PAN + ResNet18**  
Test Accuracy: 0.35  
Mean IoU: 0.14  
Mean Dice: 0.20

**PAN + EfficientnetB2**  
Test Accuracy: 0.42  
Mean IoU: 0.21  
Mean Dice: 0.33
